In [1]:
import os
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI

load_dotenv(override=True)

True

In [11]:
openai_api_key = os.getenv("OPENAI_API_KEY")
gemini_api_key = os.getenv("GOOGLE_API_KEY")

if not openai_api_key:
    print("OpenAI API Key not found")
elif not openai_api_key.startswith("sk-proj"):
    print("Invalid OpenAI API Key")
else:
    print("Valid OpenAI API Key")

if not gemini_api_key:
    print("Gemini API Ket not found")
elif not gemini_api_key.startswith("AI"):
    print("Invlaid Gemini API Key")
else:
    print("Valid Gemini API Key")

Valid OpenAI API Key
Valid Gemini API Key


In [12]:
GPT_MODEL = "gpt-4o-mini"
GEMINI_MODEL = "gemini-2.5-flash"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

openai = OpenAI()
gemini = OpenAI(base_url=gemini_url, api_key=gemini_api_key)


In [13]:
system_message = 'You are a helpful assistant'

In [6]:
#callback function to start to build memory

def chat(message, history):
    return "Bananas"


In [ ]:
gr.ChatInterface(fn=chat, type='messages').launch() #type='messages' mean gradio will use OpenAI message format

In [16]:
#define a better callback - 
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]#use for models outside of OpenAI
    messages = [{"role": "system" , "content":system_message,}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model = GEMINI_MODEL, messages=messages)
    return response.choices[0].message.content

In [17]:
gr.ChatInterface(fn=chat, type='messages').launch()


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## Prompting & RAG

In [18]:
#one shot prompt
system_message = " You are a helpful assistant in a clothes store. You should try to gently encourage\
the customer to try items that are on sale. Hats are 60% off and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat, \
You could reply with 'Wonderful - we have lots of hats. Several are include in our sales event and are up to 60% off!\
Encourage the customer to buy hats if they are unsure what to get" 
    

In [19]:
gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/alexanderpikelis/projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/alexanderpikelis/projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/alexanderpikelis/projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/alexanderpikelis/projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1621, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/alexanderpikelis/proj

In [ ]:
#further functionality hard coded - dont use in prod -> not functional for language
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=GPT_MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In reality we would use RAG to inject relevant content into the prompt instead of hardcoding it
